# Module 04 — Lecture 3: GPU Parameter Sweeps

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/module_04_hodgkin_huxley/03_parameter_sweep.ipynb)

---

One of the most powerful applications of GPU simulation: running **thousands of parameter combinations simultaneously**. A task that would take hours on a CPU completes in seconds on a GPU.

**Applications in neuroscience:**
- f-I curves with high resolution (>1000 I values)
- Sensitivity analysis (how does firing rate change with gNa?)
- Fitting models to experimental data (search parameter space in parallel)
- Phase diagrams (2D parameter sweeps)

**Learning objectives:**
- Map parameter combinations to GPU threads
- Compute the HH f-I curve with 5000-point resolution in one GPU call
- Perform a 2D phase diagram sweep (I_ext vs g_Na)
- Compare to the LIF f-I curve

In [ ]:
!nvidia-smi

In [ ]:
%%writefile hh_sweep.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); if(e!=cudaSuccess){ \
    fprintf(stderr,"CUDA: %s\n",cudaGetErrorString(e));exit(1);}} while(0)

// Standard HH params; gNa can be overridden per neuron for sweep
__constant__ float c_Cm, c_gK, c_gL, c_ENa, c_EK, c_EL, c_dt;

__device__ __forceinline__ float alpha_m(float V) {
    float d=V+40.f; return (fabsf(d)<1e-5f)?1.f:0.1f*d/(1.f-expf(-d/10.f)); }
__device__ __forceinline__ float beta_m(float V)  { return 4.f*expf(-(V+65.f)/18.f); }
__device__ __forceinline__ float alpha_h(float V) { return 0.07f*expf(-(V+65.f)/20.f); }
__device__ __forceinline__ float beta_h(float V)  { return 1.f/(1.f+expf(-(V+35.f)/10.f)); }
__device__ __forceinline__ float alpha_n(float V) {
    float d=V+55.f; return (fabsf(d)<1e-5f)?0.1f:0.01f*d/(1.f-expf(-d/10.f)); }
__device__ __forceinline__ float beta_n(float V)  { return 0.125f*expf(-(V+65.f)/80.f); }

// Each thread simulates one (I_ext, gNa) combination and counts spikes
__global__ void hh_sweep_kernel(
    const float* I_ext_arr,   // [N] input currents
    const float* gNa_arr,     // [N] Na conductances (for 2D sweep)
    int*   spike_count,       // [N] output: number of spikes
    int N, int T_steps, float T_ms_skip  // skip first T_ms_skip ms before counting
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    float I   = I_ext_arr[i];
    float gNa = gNa_arr[i];
    float dt  = c_dt;

    // Initialise at rest steady state
    float V = -65.0f;
    float am=alpha_m(V),bm=beta_m(V),ah=alpha_h(V),bh=beta_h(V),an=alpha_n(V),bn=beta_n(V);
    float m=am/(am+bm), h=ah/(ah+bh), n=an/(an+bn);

    int spikes = 0;
    int skip_steps = (int)(T_ms_skip / dt);
    int above_thresh = 0;  // track rising edge

    for (int step = 0; step < T_steps; step++) {
        // Euler step
        float INa = gNa*m*m*m*h*(V-c_ENa);
        float IK  = c_gK*n*n*n*n*(V-c_EK);
        float IL  = c_gL*(V-c_EL);
        float dV  = (I - INa - IK - IL) / c_Cm;
        float dm  = alpha_m(V)*(1.f-m) - beta_m(V)*m;
        float dh  = alpha_h(V)*(1.f-h) - beta_h(V)*h;
        float dn  = alpha_n(V)*(1.f-n) - beta_n(V)*n;
        V += dt*dV; m+=dt*dm; h+=dt*dh; n+=dt*dn;
        m=fmaxf(0.f,fminf(1.f,m)); h=fmaxf(0.f,fminf(1.f,h)); n=fmaxf(0.f,fminf(1.f,n));

        // Spike detection (rising edge through 0 mV)
        if (step >= skip_steps) {
            if (V > 0.f && !above_thresh) { spikes++; above_thresh = 1; }
            if (V < -30.f) above_thresh = 0;
        }
    }
    spike_count[i] = spikes;
}

int main() {
    float Cm=1.f,gK=36.f,gL=0.3f,ENa=50.f,EK=-77.f,EL=-54.4f,dt=0.01f;
    CUDA_CHECK(cudaMemcpyToSymbol(c_Cm,&Cm,4)); CUDA_CHECK(cudaMemcpyToSymbol(c_gK,&gK,4));
    CUDA_CHECK(cudaMemcpyToSymbol(c_gL,&gL,4)); CUDA_CHECK(cudaMemcpyToSymbol(c_ENa,&ENa,4));
    CUDA_CHECK(cudaMemcpyToSymbol(c_EK,&EK,4)); CUDA_CHECK(cudaMemcpyToSymbol(c_EL,&EL,4));
    CUDA_CHECK(cudaMemcpyToSymbol(c_dt,&dt,4));

    // 1D sweep: I from 0 to 30 μA/cm², gNa=120 (fixed)
    const int N1D = 5000;
    const float T_ms = 500.f, T_skip = 100.f;
    int T_steps = (int)(T_ms/dt);

    float *h_I=(float*)malloc(N1D*4), *h_gNa=(float*)malloc(N1D*4);
    for(int i=0;i<N1D;i++) {
        h_I[i]   = 30.f * i / (N1D-1.f);
        h_gNa[i] = 120.f;
    }

    float *dI,*dgNa; int *dsc;
    CUDA_CHECK(cudaMalloc(&dI,N1D*4)); CUDA_CHECK(cudaMalloc(&dgNa,N1D*4));
    CUDA_CHECK(cudaMalloc(&dsc,N1D*4));
    CUDA_CHECK(cudaMemcpy(dI,h_I,N1D*4,cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dgNa,h_gNa,N1D*4,cudaMemcpyHostToDevice));

    int thr=256,blk=(N1D+thr-1)/thr;
    cudaEvent_t t0,t1; float ms;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));
    CUDA_CHECK(cudaEventRecord(t0));
    hh_sweep_kernel<<<blk,thr>>>(dI,dgNa,dsc,N1D,T_steps,T_skip);
    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    CUDA_CHECK(cudaEventElapsedTime(&ms,t0,t1));

    int* hsc=(int*)malloc(N1D*4);
    CUDA_CHECK(cudaMemcpy(hsc,dsc,N1D*4,cudaMemcpyDeviceToHost));

    printf("1D sweep: N=%d, T=%.0f ms, GPU=%.2f ms\n",N1D,T_ms,ms);

    // Write f-I data
    FILE* f=fopen("fi_curve_hh.txt","w");
    float T_eff = (T_ms - T_skip) / 1000.f;
    for(int i=0;i<N1D;i++)
        fprintf(f,"%.4f %.2f\n", h_I[i], hsc[i]/T_eff);
    fclose(f);
    printf("f-I curve written to fi_curve_hh.txt\n");

    free(h_I); free(h_gNa); free(hsc);
    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    cudaFree(dI); cudaFree(dgNa); cudaFree(dsc);
    return 0;
}

In [ ]:
!nvcc -O2 -o hh_sweep hh_sweep.cu -lm && ./hh_sweep

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load and plot the HH f-I curve (5000-point resolution)
data = np.loadtxt('fi_curve_hh.txt')
I_vals = data[:, 0]
fr_hz  = data[:, 1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(I_vals, fr_hz, 'b-', lw=1.5, label='HH f-I (GPU, 5000 points)')

# Find rheobase
firing_mask = fr_hz > 0
if firing_mask.any():
    I_rheo = I_vals[firing_mask][0]
    ax.axvline(I_rheo, color='r', linestyle='--', alpha=0.7,
               label=f'Rheobase = {I_rheo:.2f} μA/cm²')

ax.set_xlabel('Input current I (μA/cm²)', fontsize=13)
ax.set_ylabel('Firing rate (Hz)', fontsize=13)
ax.set_title('Hodgkin-Huxley f-I Curve\n(5000 neurons simulated in parallel on GPU)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('hh_fi_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Max firing rate: {fr_hz.max():.0f} Hz at I = {I_vals[fr_hz.argmax()]:.1f} μA/cm²")
print(f"On CPU (serial): this sweep would take ~{5000 * 500 / 0.01 * 50 / 1e10:.0f} minutes")
print(f"On GPU: done in seconds")

## Summary

GPU parameter sweeps are a paradigm shift in computational neuroscience:

| CPU approach | GPU approach |
|-------------|-------------|
| 100 I values, 5 minutes | 5000 I values, 2 seconds |
| Coarse f-I curve, miss bifurcations | Fine resolution, see all transitions |
| 1D sweep takes hours | 2D sweep (I × gNa) still seconds |

**Key design pattern:** Thread i ↔ parameter set i. All threads run the same model with different parameters. The GPU's parallelism makes exhaustive parameter exploration practical.

**Proceed to:** [Exercise 04](exercises/ex04_stub.ipynb) — add a fast potassium A-current to the model.